**Day 3 complete — documented small finite-grid baseline.**
Code, theory notes, worked calculations and numerical verification are completed.
Prepared by Nouman with ChatGPT assistance under reported Parsa authorization.
Personal independent review by Parsa is not claimed. Original evidence is retained unchanged.
See ../audit_days01_03/README.md for the completion checklist and limitations.


# Day 3: optimization and welfare certificate
Day 3 implementation, written derivation, worked iteration, and independent-formulation numerical checks are complete for the small finite-grid baseline.

Read the short cells in order: setup, line search, one iteration, solver, audit, evidence.
Locally, open this notebook from the repository root or day03 folder.
In Colab, upload this notebook and day02/model.py to the runtime, then Run all.
This is the three-state, two-agent illustration, not the full eight-agent Example 2.


## 1. Reuse Day 2
Load the shared model from day02/model.py. In Colab, load the uploaded model.py.
The objective, gradient, and oracle remain the same; Day 3 adds the repeated optimization.


In [1]:
import importlib.util, json, sys
from pathlib import Path
import numpy as np

base = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
candidates = [base.parent / "day02/model.py", base / "day02/model.py", base / "model.py"]
model_path = next((p for p in candidates if p.is_file()), None)
if model_path is None:
    raise FileNotFoundError("Upload day02/model.py in Colab or run inside this repository.")
spec = importlib.util.spec_from_file_location("day3_model", model_path)
module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = module
spec.loader.exec_module(module)
Model, feasibility, linear_oracle = module.Model, module.feasibility, module.linear_oracle


## 2. Inputs and declared tolerances
We keep the Day 1/2 inputs: midpoint wealth and ranks, equal probabilities,
types `(h1,h2)`, and weights `(1,1)`. Settings below control this run explicitly.
Change agreed model inputs here, then rerun every cell. Source and human-review
files are not overwritten.

The gap tolerance is an absolute welfare tolerance on this finite grid.
The iteration cap is a limit, not an alternative success criterion.

In [2]:
r = (np.arange(3, dtype=float) + 0.5) / 3
model = Model(r.copy(), r.copy(), np.full(3, 1/3), np.ones(2), ("h1", "h2"))
settings = dict(gap_tol=1e-9, max_iter=2000, budget_atol=1e-10,
                domain_atol=1e-12, welfare_atol=1e-13)

def check_feasible(c):
    diagnostic = feasibility(c, model.wealth,
                             settings["budget_atol"], settings["domain_atol"])
    if not diagnostic["feasible"]: raise AssertionError(diagnostic)
    return diagnostic

initial = model.equal_sharing()
print("Initial welfare:", model.objective(initial))
print(check_feasible(initial))

Initial welfare: 1.2362178075439176
{'budget_residual': 0.0, 'minimum_consumption': 0.08333333333333333, 'minimum_increment_including_initial': 0.08333333333333333, 'feasible': True}


## 3. Line search: how far should we move?
The oracle gives a feasible candidate $V$. The direction is $D=V-C$.
Every $C+tD$, for $0\le t\le1$, is feasible: it is a mixture of two feasible points.

We maximize $F(C+tD)$. Its derivative is
$q(t)=\sum_{j,i}g_{ji}(C+tD)D_{ji}$.
For these concave log utilities, $q$ decreases. Take the endpoint if $q(1)\ge0$;
otherwise locate its zero by bisection (42 halvings, as in the guide).
If $q(0)\le0$, take a zero step. The gap is checked before searching.

We use

$$
C_{\text{new}}=(1-t)C+tV,\qquad 0\le t\le1,
$$

where \(C\) is the current allocation and \(V\) is the oracle’s allocation.

| Step \(t\) | Meaning |
| :--- | :--- |
| **0** | Keep the current allocation |
| **0.1** | Move 10% toward the oracle’s allocation |
| **1** | Accept the oracle’s allocation completely |

Because both allocations obey the constraints, their mixture also obeys them.

In [3]:
def line_search(c, direction):
    def slope(t):
        return float(np.sum(model.gradient(c + t * direction) * direction))
    if slope(0.0) <= 0: return 0.0
    if slope(1.0) >= 0: return 1.0
    left, right = 0.0, 1.0
    for _ in range(42):
        middle = (left + right) / 2
        if slope(middle) > 0: left = middle
        else: right = middle
    return (left + right) / 2

## 4. One iteration you can inspect
The oracle initially gives all consumption to the h2 agent. That maximizes the
*linear approximation*, not the log objective. Line search therefore takes only
a fraction of the proposed move. Compare these values with the explicit scalar worked iteration in DERIVATION.md and the separate high-precision reference in audit_days01_03.

In [4]:
g = model.gradient(initial)
candidate = linear_oracle(g, model.wealth)
direction = candidate - initial
first_step = line_search(initial, direction)
after_first = initial + first_step * direction
print("Oracle allocation:\n", candidate)
print("Initial gap:", float(np.sum(g * direction)))
print("Step:", first_step)
print("Welfare after update:", model.objective(after_first))
print(f"A step of approximately {first_step*100:.2f}% means that percentage of the proposed change, not of total wealth.")
print(check_feasible(after_first))

Oracle allocation:
 [[0.         0.16666667]
 [0.         0.5       ]
 [0.         0.83333333]]
Initial gap: 0.003870827167386251
Step: 0.11011256374160894
Welfare after update: 1.2364309156398132
A step of approximately 11.01% means that percentage of the proposed change, not of total wealth.
{'budget_residual': 0.0, 'minimum_consumption': 0.07415728635486592, 'minimum_increment_including_initial': 0.07415728635486592, 'feasible': True}


## 5. Repeat, log, and stop
At each current allocation, recompute the objective, gradient, oracle, and gap.
The log's welfare, gap, and bound all belong to the **same allocation**.
`step_to_next` records the step subsequently taken from that row; the final row
has no further update.

Concavity gives $F(C^*)\le F(C)+\langle g(C),C^*-C\rangle$.
Since $V$ maximizes the linear objective over feasible allocations,
$\langle g(C),C^*-C\rangle\le\langle g(C),V-C\rangle=G(C)$.
Thus $F(C)\le F(C^*)\le F(C)+G(C)$, up to numerical arithmetic.

We check every update and require welfare not to decrease beyond rounding
tolerance. A materially negative gap is an error; a tiny negative rounding
value is recorded and clipped to zero for the reported numerical bound.
The certificate covers this grid, not distribution/quadrature error, and relies
on concavity and an exact linear oracle.

In [5]:
def solve(start):
    c, log = np.array(start, copy=True), []
    for iteration in range(settings["max_iter"] + 1):
        diagnostic = check_feasible(c)
        value, gradient = model.objective(c), model.gradient(c)
        candidate = linear_oracle(gradient, model.wealth)
        check_feasible(candidate)
        direction = candidate - c
        raw_gap = float(np.sum(gradient * direction))
        if raw_gap < -1e-12: raise AssertionError("Negative oracle gap")
        gap = max(0.0, raw_gap)
        log.append(dict(iteration=iteration, welfare=value, raw_gap=raw_gap,
                        gap=gap, upper_bound=value + gap, **diagnostic,
                        step_to_next=None))
        if gap <= settings["gap_tol"]: return c, log, "gap_reached"
        if iteration == settings["max_iter"]: return c, log, "iteration_cap"
        step = line_search(c, direction)
        updated = c + step * direction
        check_feasible(updated)
        if model.objective(updated) < value - settings["welfare_atol"]:
            raise AssertionError("Welfare decreased")
        if step == 0 or np.array_equal(updated, c): return c, log, "stalled"
        log[-1]["step_to_next"] = step
        c = updated

In [6]:
allocation, log, status = solve(initial)
final = log[-1]
print("Status:", status, "| Updates:", final["iteration"])
print("F(C):", final["welfare"]) # F(C) is the welfare we have achieved.
print("G(C):", final["gap"]) # G(C) bounds how much welfare improvement could remain.
print("F(C) + G(C):", final["upper_bound"]) # is an upper bound on the optimum.
print("Allocation:\n Agent1(h1)-shares  Agent2(h2)-shares\n", allocation) # rows are the total wealth that was allocated.
# total wealth is 0.1666667 0.500000 0.83333

Status: gap_reached | Updates: 291
F(C): 1.2368115793474153
G(C): 9.597698538699184e-10
F(C) + G(C): 1.236811580307185
Allocation:
 Agent1(h1)-shares  Agent2(h2)-shares
 [[0.03063596 0.1360307 ]
 [0.21142687 0.28857313]
 [0.4030056  0.43032774]]


## 6. Independent optimizer and final oracle check
SLSQP works with direct consumption variables, explicit budget equations,
nonnegativity, and monotonicity inequalities. It uses no increment or tail-score
construction. It shares Day 2's already checked objective and gradient.
This is an independent optimization route, not an independent theory derivation.

We also compare the **final** oracle with a direct-variable HiGHS LP. Tight
HiGHS tolerances address near-tied scores late in convergence. Accept the oracle
value difference at `1e-9`; accept welfare agreement with SLSQP within the final
gap plus `1e-9` numerical allowance. Require both optimizers to be feasible.
Different allocation arrays can coexist with close welfare values.

In [7]:
from scipy.optimize import minimize, linprog

m, n = model.shape
budget_matrix = np.kron(np.eye(m), np.ones((1, n)))
monotone_matrix = np.zeros(((m-1)*n, m*n))
for j in range(m-1):
    for i in range(n):
        row = j*n+i
        monotone_matrix[row, j*n+i] = 1
        monotone_matrix[row, (j+1)*n+i] = -1

constraints = [dict(type="eq", fun=lambda x: budget_matrix @ x - model.wealth,
                    jac=lambda x: budget_matrix),
               dict(type="ineq", fun=lambda x: -monotone_matrix @ x,
                    jac=lambda x: -monotone_matrix)]
reference = minimize(lambda x: -model.objective(x.reshape(m, n)), initial.ravel(),
                     jac=lambda x: -model.gradient(x.reshape(m, n)).ravel(),
                     bounds=[(0, None)]*(m*n), constraints=constraints,
                     method="SLSQP", options=dict(ftol=1e-13, maxiter=1000))
if not reference.success: raise AssertionError(reference.message)
reference_allocation = reference.x.reshape(m, n)
reference_diagnostic = check_feasible(reference_allocation)

In [8]:
final_gradient = model.gradient(allocation)
lp = linprog(-final_gradient.ravel(), A_eq=budget_matrix, b_eq=model.wealth,
             A_ub=monotone_matrix if m > 1 else None,
             b_ub=np.zeros((m-1)*n) if m > 1 else None, bounds=(0, None),
             method="highs", options=dict(primal_feasibility_tolerance=1e-10,
                                          dual_feasibility_tolerance=1e-10))
if not lp.success: raise AssertionError(lp.message)
check_feasible(lp.x.reshape(m, n))
oracle_value = float(np.sum(final_gradient * linear_oracle(final_gradient, model.wealth)))
lp_difference = abs(oracle_value + lp.fun)
reference_welfare = model.objective(reference_allocation)
welfare_difference = abs(final["welfare"] - reference_welfare)
if lp_difference > 1e-9: raise AssertionError("Oracle/LP discrepancy")
if welfare_difference > final["gap"] + 1e-9: raise AssertionError("Solver discrepancy")
if not final["welfare"]-1e-9 <= reference_welfare <= final["upper_bound"]+1e-9:
    raise AssertionError("Reference outside certificate plus numerical allowance")
if status != "gap_reached": raise AssertionError("Solver did not reach gap tolerance")
print("Independent welfare:", reference_welfare)
print("Absolute welfare difference:", welfare_difference)
print("Final oracle/LP value difference:", lp_difference)
print("PASS: small-run computational audit; delegated Day 3 code and theory deliverables complete.")

Independent welfare: 1.236811579347413
Absolute welfare difference: 2.220446049250313e-15
Final oracle/LP value difference: 2.7755575615628914e-17
PASS: small-run computational audit; delegated Day 3 code and theory deliverables complete.


## 7. Save evidence
Every successful run saves a new timestamped results folder with all iteration
rows, both allocations, inputs, tolerances, versions, and source hashes.
The printed summary reports the finite-grid numerical certificate.
If a prior cell fails, do not use a previous successful result as the new run.

This cell does not rewrite a human review record or claim joint completion.
Save or download your runtime's results before closing Colab.

In [9]:
import csv, hashlib, platform
from datetime import datetime, timezone
from importlib.metadata import version

out = base / ("results_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
out.mkdir()
config = dict(wealth=model.wealth.tolist(), ranks=model.ranks.tolist(),
              probabilities=model.probabilities.tolist(),
              pareto_weights=model.pareto_weights.tolist(), agent_types=list(model.agent_types))
summary = dict(status=status, deliverables_status="complete for the small finite-grid baseline",
               personal_independent_review="not claimed",
               final=final, maximum_budget_residual=max(row["budget_residual"] for row in log),
               minimum_increment_over_run=min(row["minimum_increment_including_initial"] for row in log),
               reference_welfare=reference_welfare, welfare_difference=welfare_difference,
               maximum_allocation_difference=float(np.max(abs(allocation-reference_allocation))),
               final_oracle_lp_difference=float(lp_difference),
               certificate_scope="concave finite-grid optimization; floating-point accuracy")
for name, data in [("summary", summary), ("config_used", config), ("settings", settings)]:
    (out / (name + ".json")).write_text(json.dumps(data, indent=2, allow_nan=False))
with (out / "iterations.csv").open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(log[0]))
    writer.writeheader(); writer.writerows(log)
np.savetxt(out / "allocations.csv", np.column_stack((model.wealth, model.ranks,
           model.probabilities, allocation, reference_allocation)), delimiter=",",
           header="wealth,rank,probability," + ",".join(
               [f"solver_c{i+1}" for i in range(n)] + [f"reference_c{i+1}" for i in range(n)]), comments="")

In [10]:
environment = dict(python=sys.version, platform=platform.platform(),
                   numpy=version("numpy"), scipy=version("scipy"))
source_hashes = {"model.py": hashlib.sha256(model_path.read_bytes()).hexdigest()}
for source in ("Risk_Sharing_Day_3.ipynb", "day3_example.py"):
    if (base / source).exists():
        source_hashes[source] = hashlib.sha256((base / source).read_bytes()).hexdigest()
(out / "environment.json").write_text(json.dumps(environment, indent=2))
(out / "source_hashes.json").write_text(json.dumps(source_hashes, indent=2))
print("Evidence:", out.resolve())
print("Budget residual:", summary["maximum_budget_residual"])
print("Minimum increment:", summary["minimum_increment_over_run"])
print("Day 3 complete for the small finite-grid baseline. See audit_days01_03 for verification and scope.")

Evidence: /workspace/scratch/b0a5f4db9d25/repo/day03/results_20261013T153058840371Z
Budget residual: 3.3306690738754696e-16
Minimum increment: 0.030635963397630247
Day 3 complete for the small finite-grid baseline. See audit_days01_03 for verification and scope.


### Results archive contains:

| File | Purpose |
|---|---|
| `iterations.csv` | Welfare, gap, bound, feasibility, and steps throughout the run |
| `allocations.csv` | Our allocation and the independent optimizer’s allocation |
| `summary.json` | Final results and audit measurements |
| `config_used.json` | Exact model inputs |
| `settings.json` | Tolerances and iteration limit |
| `environment.json` | Python, NumPy, and SciPy versions |
| `source_hashes.json` | Recorded model-code fingerprint |

There are 292 log rows: the initial allocation plus the allocations after 291 updates.
This row count is for the default small run; changing inputs or tolerances can change it.


## Completed scope and next stage
Day 3 code, certificate proof, worked iteration and analytic small-grid verification are complete. Day 4 is the full eight-agent replication. The supplied replication code package was not provided; this uses our independent Day 2 implementation.

**Sources:** your uploaded Day 2 notebook; five-day research plan, Day 3 on p. 3;
replication guide, sections 5.4–5.5 on pp. 6–7.
This notebook makes no continuous-model error claim and is not a nonconcave solver.